# Определение ботов по событиям Avito

Ноутбук повторяет основное решение из `solution.py`: удаляет точные дубли событий, строит признаки поведения, проверяет модель на более поздних датах и создаёт итоговый файл. Код расчёта признаков и обучения находится в одном месте — в ноутбуке мы используем те же функции, что и в командном запуске.

## Идея решения

События рассматриваются только внутри окна наблюдения каждой cookie: `window_start_ts <= event_ts < window_end_ts`. По ним считаются частота и типы действий, интервалы между событиями, разнообразие объектов и категорий, характеристики указателя и User-Agent. Эти признаки подаём в CatBoost.

Тест относится к более позднему периоду, поэтому качество проверяем не случайным разбиением, а на последних трёх днях обучающей выборки. Так проверка лучше похожа на применение модели к будущим данным. Основная метрика — precision при recall не ниже 70%.

## Подготовка

Положите `train.csv`, `test.csv` и `events.csv.gz` в папку `data/` в корне проекта и установите зависимости командой `python -m pip install -r requirements.txt`. Запускайте ячейки по порядку.

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd

from solution import (
    build_features,
    evaluate_models,
    load_data,
    train_final,
)

## Загружаем данные и убираем точные дубли

Удаляем только полностью совпадающие строки событий. Затем объединяем обучающие и тестовые метаданные, чтобы рассчитать признаки одинаковым способом для обеих выборок.

In [ ]:
train, test, events = load_data()
duplicate_count = int(events.duplicated().sum())
events = events.drop_duplicates().copy()

print(f"Обучающих строк: {len(train):,}")
print(f"Тестовых строк: {len(test):,}")
print(f"Удалено точных дублей событий: {duplicate_count:,}")
train.head()

## Сначала посмотрим на данные

На графиках видно, сколько в обучении ботов и обычных cookie, как примеры распределены по датам и где начинается отложенный период.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
colors = ["#4c78a8", "#f58518"]

class_counts = train.target.value_counts().reindex([0, 1], fill_value=0)
bars = axes[0].bar(["Обычные cookie", "Боты"], class_counts.values, color=colors, width=0.62)
axes[0].set_title("Классы в обучающей выборке", loc="left", weight="bold")
axes[0].set_ylabel("Количество cookie")
axes[0].spines[["top", "right"]].set_visible(False)
axes[0].bar_label(bars, fmt="%d", padding=4)

daily_rows = train.assign(day=train.window_start_ts.dt.floor("D")).groupby("day").size()
axes[1].plot(daily_rows.index, daily_rows.values, color="#4c78a8", linewidth=2.5, marker="o")
axes[1].axvline(pd.Timestamp("2026-04-17"), color="#f58518", linestyle="--", label="Начало проверки")
axes[1].set_title("Распределение примеров по датам", loc="left", weight="bold")
axes[1].set_ylabel("Количество cookie")
axes[1].tick_params(axis="x", rotation=35)
axes[1].legend(frameon=False)
axes[1].spines[["top", "right"]].set_visible(False)

fig.tight_layout()
plt.show()

## Строим признаки

`build_features` возвращает по одной строке на cookie. Функция сама ограничивает события окном наблюдения, рассчитывает числовые и категориальные признаки и сохраняет порядок исходных cookie.

In [ ]:
metadata = pd.concat([train.drop(columns="target"), test], ignore_index=True)
all_features = build_features(metadata, events)
train_features = all_features.reindex(train.cookie_id)
test_features = all_features.reindex(test.cookie_id)

print(f"Cookie: {len(all_features):,}")
print(f"Признаков: {all_features.shape[1]:,}")
all_features.head()

## Проверяем качество на будущих датах

Валидация обучает CatBoost на ранних датах и проверяет его на последних трёх днях train. Дополнительно рассчитываются справочные результаты других моделей и их смеси; для итогового файла используется CatBoost.

In [ ]:
validation_results = evaluate_models(train, train_features)
print(json.dumps(validation_results, indent=2, ensure_ascii=False))

In [ ]:
model_names = {
    "catboost": "CatBoost",
    "extra_trees": "Extra Trees",
    "hist_gradient_boosting": "Hist. Gradient Boosting",
    "best_rank_blend": "Смесь моделей",
}
metric_names = {
    "precision_at_recall_070": "Precision при Recall ≥ 70%",
    "average_precision": "Average Precision",
    "roc_auc": "ROC-AUC",
}
model_results = {
    label: validation_results[key]
    for key, label in model_names.items()
    if key in validation_results and validation_results[key] is not None
}

fig, ax = plt.subplots(figsize=(11, 5))
x = range(len(model_results))
width = 0.24
palette = ["#4c78a8", "#72b7b2", "#f58518"]
for offset, (metric, label), color in zip([-width, 0, width], metric_names.items(), palette):
    values = [result[metric] for result in model_results.values()]
    ax.bar([position + offset for position in x], values, width, label=label, color=color)

ax.set_xticks(list(x), model_results.keys(), rotation=12, ha="right")
ax.set_ylim(0, 1)
ax.set_ylabel("Значение метрики")
ax.set_title("Сравнение моделей на отложенных датах", loc="left", weight="bold")
ax.legend(frameon=False, ncols=1, loc="upper left", bbox_to_anchor=(1.01, 1))
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
plt.show()

## Обучаем итоговую модель и сохраняем файл

После выбора числа итераций модель заново обучается на всём train и рассчитывает оценки для test. Полученный `submission.csv` можно отправлять на платформу.

In [ ]:
iterations = max(300, int(validation_results["catboost"]["best_iteration"] * 1.15))
submission = train_final(
    train,
    test,
    train_features,
    test_features,
    iterations=iterations,
)

print(f"Сохранён submission.csv: {len(submission):,} строк")
submission.head()

In [ ]:
importance = pd.read_csv("feature_importance.csv").nlargest(15, "importance").sort_values("importance")
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(importance.feature, importance.importance, color="#4c78a8")
ax.set_title("Какие признаки модель использует чаще", loc="left", weight="bold")
ax.set_xlabel("Важность признака")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", alpha=0.2)
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(submission.score, bins=30, color="#72b7b2", edgecolor="white")
ax.set_title("Распределение оценок для тестовых cookie", loc="left", weight="bold")
ax.set_xlabel("Оценка модели")
ax.set_ylabel("Количество cookie")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()